# 🎙️ Hausa Tonal VITS: End-to-End Smoke Training
**Dataset**: [suleiman2003/W_hausa_v7](https://huggingface.co/datasets/suleiman2003/W_hausa_v7)
**GitHub Repository**: [issuleiman/hausa-tonal-vits](https://github.com/issuleiman/hausa-tonal-vits)

This notebook runs an end-to-end smoke training experiment on Kaggle (GPU accelerator recommended: T4 x2 or P100):
1. **Single-Speaker Training**: Speaker ID `88` (100 samples)
2. **Multi-Speaker Training**: Speaker IDs `79` & `247` (50 samples each = 100 samples)
3. **Validation Evaluation**: Computes MCD, F0 RMSE, and RTF metrics
4. **Synthesis & Playback**: Generates speech audio for all speakers with an interactive audio player

In [ ]:
# ==============================================================================
# 1. Install Framework and Dependencies
# ==============================================================================
!pip install -q git+https://github.com/issuleiman/hausa-tonal-vits.git
!pip install -q datasets soundfile scipy pyyaml

import hausa_tts
print("✓ Hausa Tonal VITS loaded from:", hausa_tts.__file__)

In [ ]:
# ==============================================================================
# 2. Stream Dataset from HuggingFace (100 samples each)
# ==============================================================================
import os, time, soundfile as sf, numpy as np, torch
from pathlib import Path
from datasets import load_dataset
from hausa_tts import normalise
from hausa_tts.data import _guess_utt, resample_linear

TARGET_SR = 22050
single_dir = Path("data_single_speaker")
multi_dir = Path("data_multi_speaker")
(single_dir / "wavs").mkdir(parents=True, exist_ok=True)
(multi_dir / "wavs").mkdir(parents=True, exist_ok=True)

def extract_audio_array(audio_data, target_sr=22050):
    arr, sr = None, target_sr
    if hasattr(audio_data, "get_all_samples"):
        samples = audio_data.get_all_samples()
        arr, sr = samples.data, getattr(samples, "sample_rate", target_sr)
    elif isinstance(audio_data, dict):
        if "array" in audio_data and audio_data["array"] is not None:
            arr, sr = audio_data["array"], audio_data.get("sampling_rate", target_sr)
        elif "bytes" in audio_data and audio_data["bytes"] is not None:
            import io
            arr, sr = sf.read(io.BytesIO(audio_data["bytes"]), dtype="float32")
    elif hasattr(audio_data, "data"):
        arr, sr = audio_data.data, getattr(audio_data, "sample_rate", target_sr)
    if arr is None: return None, sr
    if isinstance(arr, torch.Tensor): arr = arr.detach().cpu().float().numpy()
    elif not isinstance(arr, np.ndarray): arr = np.array(arr, dtype=np.float32)
    arr = arr.squeeze()
    if arr.ndim > 1: arr = arr.mean(axis=0 if arr.shape[0] < arr.shape[-1] else -1)
    return arr.astype(np.float32), int(sr)

print("Connecting to suleiman2003/W_hausa_v7 stream...")
ds = load_dataset("suleiman2003/W_hausa_v7", split="train", streaming=True)

collected_single = []
collected_multi = {79: [], 247: []}

for item in ds:
    spk_id = int(item.get("speaker_id", -1))
    text = item.get("text", "").strip()
    audio_data = item.get("audio", None)
    if not text or audio_data is None:
        continue

    clean_text = normalise(text)
    if not clean_text or len(clean_text) < 3:
        continue

    arr, sr = extract_audio_array(audio_data, TARGET_SR)
    if arr is None or len(arr) == 0:
        continue
    if sr != TARGET_SR:
        arr = resample_linear(arr, sr, TARGET_SR)

    dur = len(arr) / float(TARGET_SR)
    if dur < 0.6 or dur > 12.0:
        continue

    utt = _guess_utt(clean_text)

    # Single speaker: 88 (100 samples)
    if spk_id == 88 and len(collected_single) < 100:
        idx = len(collected_single)
        p = single_dir / "wavs" / f"spk88_{idx:04d}.wav"
        sf.write(str(p), arr, TARGET_SR)
        collected_single.append(f"{p.as_posix()}|{clean_text}|{utt}||0")
        if len(collected_single) % 25 == 0:
            print(f"  [Single 88] {len(collected_single)}/100 samples collected")

    # Multi speaker: 79 & 247 (50 samples each)
    if spk_id in collected_multi and len(collected_multi[spk_id]) < 50:
        sid_idx = 0 if spk_id == 79 else 1
        idx = len(collected_multi[spk_id])
        p = multi_dir / "wavs" / f"spk{spk_id}_{idx:04d}.wav"
        sf.write(str(p), arr, TARGET_SR)
        collected_multi[spk_id].append(f"{p.as_posix()}|{clean_text}|{utt}||{sid_idx}")
        if len(collected_multi[spk_id]) == 50:
            print(f"  [Multi {spk_id}] 50/50 samples collected")

    if len(collected_single) >= 100 and all(len(v) >= 50 for v in collected_multi.values()):
        break

# Save 90/10 train/val splits
def save_splits(items, d):
    val, train = items[:10], items[10:]
    with open(d / "train.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(train))
    with open(d / "val.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(val))

save_splits(collected_single, single_dir)
save_splits(collected_multi[79] + collected_multi[247], multi_dir)
print("✓ Datasets ready for training!")

In [ ]:
# ==============================================================================
# 3. Run Single-Speaker Training (Speaker 88)
# ==============================================================================
import torch
from hausa_tts import HausaVITSConfig, TrainConfig
from hausa_tts.train import HausaTTSTrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

cfg_single = HausaVITSConfig.single_speaker(batch_size=4, device=device, epochs=10)
tcfg_single = TrainConfig(
    data_dir="data_single_speaker",
    train_list="data_single_speaker/train.txt",
    val_list="data_single_speaker/val.txt",
    out_dir="runs/single_speaker_88",
    log_interval=5,
    save_interval=25,
    fp16=(device == "cuda")
)

trainer_single = HausaTTSTrainer(cfg_single, tcfg_single)
trainer_single.train(max_steps=25)
print("✓ Single-speaker smoke training complete!")

In [ ]:
# ==============================================================================
# 4. Single-Speaker Evaluation & Audio Playback
# ==============================================================================
from hausa_tts import HausaTTS, SynthOptions, save_wav
from IPython.display import Audio, display

ckpt_single = f"runs/single_speaker_88/G_final_{trainer_single.step}.pth"
tts_single = HausaTTS.from_pretrained(ckpt_single, device=device)

test_sentence = "Yaushe za ka dawo gida?"
audio_single = tts_single.speak(test_sentence, opts=SynthOptions(rate=1.0, pitch=0.0))
save_wav(audio_single, "output_single_spk88.wav", sample_rate=22050)

print(f"Synthesized: {test_sentence}")
display(Audio(audio_single, rate=22050))

In [ ]:
# ==============================================================================
# 5. Run Multi-Speaker Training (Speakers 79 & 247)
# ==============================================================================
cfg_multi = HausaVITSConfig.multi_speaker(n_speakers=2, batch_size=4, device=device, epochs=10)
tcfg_multi = TrainConfig(
    data_dir="data_multi_speaker",
    train_list="data_multi_speaker/train.txt",
    val_list="data_multi_speaker/val.txt",
    out_dir="runs/multi_speaker_79_247",
    log_interval=5,
    save_interval=25,
    fp16=(device == "cuda")
)

trainer_multi = HausaTTSTrainer(cfg_multi, tcfg_multi)
trainer_multi.train(max_steps=25)
print("✓ Multi-speaker smoke training complete!")

In [ ]:
# ==============================================================================
# 6. Multi-Speaker Evaluation & Audio Playback
# ==============================================================================
ckpt_multi = f"runs/multi_speaker_79_247/G_final_{trainer_multi.step}.pth"
spk_map = {"Speaker_79": 0, "Speaker_247": 1}
tts_multi = HausaTTS.from_pretrained(ckpt_multi, device=device, speakers=spk_map)

test_multi = "Barka da yamma, yaya aiki da gajiya?"
for spk_name in ["Speaker_79", "Speaker_247"]:
    audio = tts_multi.speak(test_multi, opts=SynthOptions(speaker=spk_name))
    save_wav(audio, f"output_{spk_name}.wav", sample_rate=22050)
    print(f"\nSynthesized [{spk_name}]: {test_multi}")
    display(Audio(audio, rate=22050))